# Merging the Airbnb tables with pandas

The right `merge` for every pair of Airbnb tables: which key, and why.

| Left table | Right table | Key | Why |
|---|---|---|---|
| `calendar` | `listings` | `listing_id` = `id` | both name the same listing; the calendar has one row per night |
| `reviews_monthly` | `listings` | `listing_id` = `id` | both name the same listing; one row per month with reviews |
| `reviews_monthly` | `weather_daily` | `month` | both describe a month in Berlin; the weather is daily, so group it by month first |
| `listings` | `neighbourhoods` | `neighbourhood` | the name of the area on the map |

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Load the tables

In [2]:
import json

import pandas as pd

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")                # one row per listing
calendar = pd.read_parquet("case-study/data/airbnb/calendar.parquet")                # one row per listing and night
reviews_monthly = pd.read_parquet("case-study/data/airbnb/reviews_monthly.parquet")  # one row per listing and month
weather_daily = pd.read_parquet("case-study/data/airbnb/weather_daily.parquet")      # one row per day

with open("case-study/data/airbnb/neighbourhoods.geojson") as file:
    areas_geojson = json.load(file)
neighbourhoods = pd.DataFrame([feature["properties"] for feature in areas_geojson["features"]])  # one row per area

## The merge syntax

```python
pd.merge(left=left_table, right=right_table, left_on="key_in_left", right_on="key_in_right", how="left")
pd.merge(left=left_table, right=right_table, on="key", how="left")   # when the key has the same name in both tables
```

`left=` is the first table, `right=` the second. `how="left"` keeps every row of the left table; `how="inner"` (the default) keeps only rows whose key is in both tables.

## Calendar and listings

Key: `listing_id` in `calendar` = `id` in `listings`. Each night gets the district and room type of its listing.

In [3]:
calendar_with_listing = pd.merge(
    left=calendar,
    right=listings[["id", "district", "room_type"]],
    left_on="listing_id", right_on="id",
)
calendar_with_listing.head()

,listing_id,date,available,minimum_nights,id,district,room_type
0,3176,2026-07-03,False,63,3176,Pankow,Entire home/apt
1,3176,2026-07-04,False,63,3176,Pankow,Entire home/apt
2,3176,2026-07-05,False,63,3176,Pankow,Entire home/apt
3,3176,2026-07-06,False,63,3176,Pankow,Entire home/apt
4,3176,2026-07-07,False,63,3176,Pankow,Entire home/apt


## Reviews and listings

Key: `listing_id` in `reviews_monthly` = `id` in `listings`. Each month of reviews gets the district of its listing.

In [4]:
reviews_with_listing = pd.merge(
    left=reviews_monthly,
    right=listings[["id", "district"]],
    left_on="listing_id", right_on="id",
)
reviews_with_listing.head()

,listing_id,month,n_reviews,id,district
0,3176,2009-06-01,1,3176,Pankow
1,3176,2010-11-01,2,3176,Pankow
2,3176,2010-12-01,1,3176,Pankow
3,3176,2011-01-01,3,3176,Pankow
4,3176,2011-02-01,1,3176,Pankow


## Listings and reviews: one row per listing

To add reviews to `listings` and keep one row per listing, sum the reviews per listing first, then merge. `how="left"` keeps listings without reviews; `fillna(0)` gives them 0.

In [5]:
reviews_per_listing = reviews_monthly.groupby("listing_id", as_index=False)["n_reviews"].sum()

listings_with_reviews = pd.merge(
    left=listings[["id", "district"]],
    right=reviews_per_listing,
    left_on="id", right_on="listing_id",
    how="left",
)
listings_with_reviews["n_reviews"] = listings_with_reviews["n_reviews"].fillna(0)
listings_with_reviews.head()

,id,district,listing_id,n_reviews
0,3176,Pankow,3176.0,150.0
1,9991,Pankow,9991.0,7.0
2,14325,Pankow,14325.0,26.0
3,17904,Friedrichshain-Kreuzberg,17904.0,298.0
4,20858,Pankow,20858.0,145.0


## Reviews and weather

Key: `month`. The weather has one row per day, so first group it by month; `.dt.to_period("M").dt.to_timestamp()` writes each date as the first day of its month, as in `reviews_monthly`. Then sum the reviews of all listings per month.

In [6]:
weather_monthly = (
    weather_daily
    .assign(month=weather_daily["date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month", as_index=False)
    .agg(mean_temperature=("temperature_2m_mean", "mean"))
)
reviews_per_month = reviews_monthly.groupby("month", as_index=False)["n_reviews"].sum()

reviews_and_weather = pd.merge(left=reviews_per_month, right=weather_monthly, on="month")
reviews_and_weather.head()

,month,n_reviews,mean_temperature
0,2016-01-01,781,-0.354839
1,2016-02-01,777,3.655172
2,2016-03-01,960,4.358065
3,2016-04-01,1048,8.700000
4,2016-05-01,1286,15.648387


## Listings and neighbourhoods

Key: `neighbourhood`, the same name in both tables, so `on=` is enough. Each listing gets the district from the map (`neighbourhood_group`).

In [7]:
listings_with_area = pd.merge(
    left=listings[["id", "neighbourhood"]],
    right=neighbourhoods,
    on="neighbourhood",
    how="left",
)
listings_with_area.head()

,id,neighbourhood,neighbourhood_group
0,3176,Prenzlauer Berg Südwest,Pankow
1,9991,Prenzlauer Berg Südwest,Pankow
2,14325,Prenzlauer Berg Nordwest,Pankow
3,17904,südliche Luisenstadt,Friedrichshain-Kreuzberg
4,20858,Prenzlauer Berg Südwest,Pankow


## Further reading

- pandas user guide: [Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html).
- The files and their links: [case-study/data/airbnb/PROVENANCE.md](../../../case-study/data/airbnb/PROVENANCE.md).